# 07 — 入出力の生データを見る

上からセルを実行すると、最後のセルに対話ビューアが表示されます。追加のPythonライブラリは不要です（既存のJupyter/IPython環境を使用）。

- **Sol / Terra / Luna**：06の統合records。各条件10件、各モデル320件。
- **GPT-3.5**：03_1のJoint質問records。N30のうち各条件の `ordinal=0〜9` の10件を表示。
- モデル・タスク・予測対象・Hをプルダウンで選択し、「前へ」「次へ」で同条件の10件を順に閲覧します。モデルや条件を変更してもordinalを維持します。
- プロンプトと回答は改行を保って表示。条件・文字列・数値等は別の表に表示し、入れ子のデータは折りたたみで確認できます。
- SolのAPI失敗6件も含みます。`ignored` は応答があるが採点用parserで解釈できなかった状態です。
- 元のJSONLには一切書き込みません。API呼び出しもありません。表示セルを再実行すれば選択状態は初期化されます。

**表示されない場合**：Pythonカーネルで上からセルを実行してください。対話部分にはJavaScriptを使うため、静的プレビューでは操作できません。信頼済みのNotebook出力を表示できるJupyterLab / Notebookを使用してください。

出力を保存するとプロンプト・応答全体がNotebookにも埋め込まれ、サイズが大きくなります。保存・共有時は必要に応じて「すべての出力をクリア」してください。配布時のこのNotebookは出力を空にしています。

表の型は表示時の値から判定したJSON相当の型です（整数値の数値はintと表示）。生のJSON表記は折りたたみの「このqueryの全フィールド」で確認できます。


In [ ]:
from pathlib import Path
import json
import hashlib
import html
from IPython.display import HTML, display

# Notebookフォルダ、リポジトリ直下、親のMatsuoフォルダから実行可能。
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for parent in candidates
             for p in (parent, parent / "mental-modeling-openai")
             if (p / "experiments/06_new_models_n10/results").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("mental-modeling-openai内でこのNotebookを開いてください。")

SOURCES = {
    "new_models": ROOT / "experiments/06_new_models_n10/results/merged/c189af3036db4adfb9e4da5bb632ea48/records.jsonl",
    "gpt35_joint": ROOT / "experiments/03_1_gpt35_history_n30_joint/results/records.jsonl",
}
records = []
source_hashes = {}
for source_name, path in SOURCES.items():
    content = path.read_bytes()  # 読み取りのみ。元ファイルへの保存処理はない。
    source_hashes[path] = hashlib.sha256(content).hexdigest()
    source_rows = [json.loads(line) for line in content.decode("utf-8").splitlines() if line.strip()]
    for record in source_rows:
        if 0 <= record["ordinal"] < 10:
            records.append({**record, "viewer_source": str(path.relative_to(ROOT))})
    print(f"{source_name}: 元データ {len(source_rows)} 行 / 表示 {sum(0 <= r['ordinal'] < 10 for r in source_rows)} 行")

from collections import Counter
counts = Counter((r["model_alias"], r["task"], r["metric"], r["H"]) for r in records)
assert len(records) == 1280 and len(counts) == 128 and set(counts.values()) == {10}, "条件・件数を確認してください。"
assert len({r["query_id"] for r in records}) == len(records), "query_idが重複しています。"
print("4モデル × 32条件 × 10件 = 1,280件。失敗行も表示対象です。")


In [ ]:
# json.loadsでJSONの\nは実際の改行に復元済み。
# pre-wrap + textContentで改行を保ち、プロンプト内のHTMLを実行せず表示する。
VIEWER_TEMPLATE = '<!doctype html>\n<html lang="ja"><head><meta charset="utf-8"><style>\n*{box-sizing:border-box}body{margin:0;padding:22px;font:14px/1.6 system-ui,sans-serif;color:#172333;background:#f4f7fb}h2{margin:0 0 6px}h3{font-size:15px;margin:0 0 10px}.muted{color:#576579} .controls,.panel{background:white;border:1px solid #dce3ed;border-radius:10px;padding:16px;margin:14px 0}.filters{display:flex;flex-wrap:wrap;gap:12px}label{display:flex;flex-direction:column;gap:4px;font-weight:600}select,button{font:inherit;padding:7px 10px;border:1px solid #b9c6d8;border-radius:6px;background:white;color:#172333}button{cursor:pointer}button:disabled{opacity:.4;cursor:default}.nav{display:flex;gap:12px;align-items:center;flex-wrap:wrap;margin-top:16px}.status{font-weight:700}.failed{color:#b42318}pre{white-space:pre-wrap;overflow-wrap:anywhere;font:13px/1.65 ui-monospace,monospace;margin:0;max-height:540px;overflow:auto}table{border-collapse:collapse;width:100%;font-size:13px}th,td{border-bottom:1px solid #e3e8ef;text-align:left;padding:7px 9px;vertical-align:top;overflow-wrap:anywhere}th{width:260px}td pre{max-height:160px}.table-scroll{max-height:480px;overflow:auto}summary{cursor:pointer;font-weight:600}details pre{margin-top:10px}.pill{padding:4px 10px;background:#eef3ff;border-radius:5px}#empty{color:#b42318}\n</style></head><body>\n<h2>Raw input / output viewer</h2><div class="muted">Joint質問 • 各条件 ordinal 0〜9 • 閲覧専用。モデルを変更しても同じsample番号を維持します。</div>\n<div class="controls"><div class="filters">\n<label>モデル<select id="model"></select></label><label>タスク<select id="task"></select></label><label>予測対象<select id="metric"></select></label><label>履歴長 H<select id="history"></select></label>\n</div><div class="nav"><button id="previous">← 前へ</button><span id="position" class="pill"></span><button id="next">次へ →</button><span id="status" class="status"></span></div><div id="empty"></div></div>\n<div id="content"></div>\n<script id="dataset" type="application/json">__DATA__</script>\n<script>\n\'use strict\';\nconst records=JSON.parse(document.getElementById(\'dataset\').textContent);\nconst $=id=>document.getElementById(id);\nconst fields={model:\'model_alias\',task:\'task\',metric:\'metric\',history:\'H\'};\nconst modelNames={\'3.5\':\'GPT-3.5 (Joint)\',sol:\'Sol\',terra:\'Terra\',luna:\'Luna\'};\nfor(const [id,key] of Object.entries(fields)){\n let values=[...new Set(records.map(r=>r[key]))];\n if(id===\'model\')values=[\'sol\',\'terra\',\'luna\',\'3.5\'].filter(v=>values.includes(v));\n else if(id===\'history\')values.sort((a,b)=>a-b);else values.sort();\n for(const value of values){const option=document.createElement(\'option\');option.value=value;option.textContent=id===\'model\'?modelNames[value]:value;$(id).append(option)}\n $(id).addEventListener(\'change\',render);\n}\nlet sampleOrdinal=0;\nfunction selected(){return records.filter(r=>Object.entries(fields).every(([id,key])=>String(r[key])===$(id).value)).sort((a,b)=>a.ordinal-b.ordinal)}\nfunction element(tag,text){const e=document.createElement(tag);if(text!==undefined)e.textContent=text;return e}\nfunction panel(title,text){const box=element(\'section\');box.className=\'panel\';box.append(element(\'h3\',title),element(\'pre\',text));return box}\nfunction details(title,value){const box=element(\'details\');box.className=\'panel\';box.append(element(\'summary\',title),element(\'pre\',JSON.stringify(value,null,2)));return box}\nfunction render(){\n const rows=selected();let index=rows.findIndex(r=>r.ordinal===sampleOrdinal);if(index<0){index=0;sampleOrdinal=rows[0]?.ordinal??0}\n $(\'content\').replaceChildren();$(\'empty\').textContent=rows.length?\'\':\'該当するデータはありません。\';\n $(\'previous\').disabled=index<=0;$(\'next\').disabled=index>=rows.length-1||!rows.length;\n $(\'position\').textContent=rows.length?`${index+1} / ${rows.length}  ·  ordinal ${sampleOrdinal}`:\'0 / 0\';\n if(!rows.length){$(\'status\').textContent=\'\';return}\n const r=rows[index];$(\'status\').textContent=r.status===\'failed\'?\'API失敗（回答なし）\':r.status===\'ignored\'?\'応答あり / 採点parserは解釈不可\':`応答あり / 採点: ${r.status}`;$(\'status\').className=\'status\'+(r.status===\'failed\'?\' failed\':\'\');\n const messages=r.request?.messages??[{role:\'system\',content:r.system_prompt},{role:\'user\',content:r.user_prompt}];\n for(const [i,m] of messages.entries())$(\'content\').append(panel(`入力 ${i+1} — ${m.role}`,typeof m.content===\'string\'?m.content:JSON.stringify(m.content,null,2)));\n const answer=r.assistant_text??r.raw_response?.choices?.[0]?.message?.content;\n $(\'content\').append(panel(\'出力 — assistant_text\',answer??\'回答は保存されていません。下の例外・API応答情報を確認してください。\'));\n const metadata=element(\'section\');metadata.className=\'panel\';metadata.append(element(\'h3\',\'条件・識別子・時間・token等（プロンプトとは別表示）\'));\n const scroll=element(\'div\');scroll.className=\'table-scroll\';const table=element(\'table\');const head=element(\'tr\');for(const t of [\'項目\',\'型\',\'値\'])head.append(element(\'th\',t));table.append(head);\n const excluded=new Set([\'system_prompt\',\'user_prompt\',\'assistant_text\']);\n for(const [key,value] of Object.entries(r)){\n  if(excluded.has(key)||(value!==null&&typeof value===\'object\'))continue;\n  const tr=element(\'tr\');tr.append(element(\'th\',key));const type=value===null?\'null\':typeof value===\'number\'?(Number.isInteger(value)?\'int\':\'float\'):typeof value===\'boolean\'?\'bool\':\'str\';tr.append(element(\'td\',type));const td=element(\'td\');td.append(element(\'pre\',String(value)));tr.append(td);table.append(tr);\n }\n scroll.append(table);metadata.append(scroll);$(\'content\').append(metadata);\n if(r.exception)$(\'content\').append(details(\'例外 — exception\',r.exception));\n const requestOptions=Object.fromEntries(Object.entries(r.request??{}).filter(([k])=>k!==\'messages\'));\n $(\'content\').append(details(\'API送信設定（messages以外）\',requestOptions));\n for(const key of [\'usage\',\'score\',\'prediction\',\'ground_truth\',\'attempts\',\'raw_response\'])if(key in r)$(\'content\').append(details(key,r[key]));\n $(\'content\').append(details(\'このqueryの全フィールド（JSON）\',r));\n}\n$(\'previous\').addEventListener(\'click\',()=>{const rows=selected();const i=rows.findIndex(r=>r.ordinal===sampleOrdinal);if(i>0){sampleOrdinal=rows[i-1].ordinal;render()}});\n$(\'next\').addEventListener(\'click\',()=>{const rows=selected();const i=rows.findIndex(r=>r.ordinal===sampleOrdinal);if(i<rows.length-1){sampleOrdinal=rows[i+1].ordinal;render()}});\nrender();\n</script></body></html>'
payload = json.dumps(records, ensure_ascii=False).replace("<", "\\u003c").replace(">", "\\u003e").replace("&", "\\u0026")
viewer_document = VIEWER_TEMPLATE.replace("__DATA__", payload)
display(HTML('<iframe title="Raw data viewer" sandbox="allow-scripts" '
             'style="width:100%;height:1250px;border:1px solid #dce3ed;border-radius:10px" '
             'srcdoc="' + html.escape(viewer_document, quote=True) + '"></iframe>'))


## 元データの変更がないことを確認（任意）

In [ ]:
# 任意：閲覧対象の元ファイルが変わっていないことを確認。
for path, expected in source_hashes.items():
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, f"元データが変更されています: {path}"
print("読み込み時と同じSHA-256です。元データに変更はありません。")
